# Notebook 2: Retrieval Failure Taxonomy

**Goal:** Classify why retrieval fails on low-scoring queries.  
**Failure types:**
- `wrong_chunk` — lexical overlap pulled an irrelevant chunk
- `right_topic_wrong_detail` — correct product, wrong specific fact
- `information_split` — answer spans chunk boundary, neither chunk alone is sufficient
- `query_ambiguity` — query matches multiple products/contexts
- `vocabulary_mismatch` — query uses different terms than source text

**Outputs:** `retrieval_failures.json` (consumed by dashboard failure analysis view)

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from analysis.utils import load_results, results_to_query_df, apply_plot_style, save_output
apply_plot_style()

In [ ]:
results = load_results()
query_df = results_to_query_df(results)
print(f'Total query-experiment pairs: {len(query_df)}')
query_df[['experiment_id', 'query_id', 'context_precision', 'context_recall']].head()

## 1. Identify Failing Queries (low precision or recall)

In [ ]:
FAILURE_THRESHOLD = 0.5
failing = query_df[
    (query_df['context_precision'] < FAILURE_THRESHOLD) |
    (query_df['context_recall'] < FAILURE_THRESHOLD)
].copy()

print(f'Failing queries: {len(failing)} / {len(query_df)} ({100*len(failing)/len(query_df):.1f}%)')
failing[['experiment_id','query_id','context_precision','context_recall']].head(10)

## 2. Heuristic Failure Classification

In [ ]:
def token_overlap(a: str, b: str) -> float:
    ta = set(re.findall(r'\w+', a.lower()))
    tb = set(re.findall(r'\w+', b.lower()))
    return len(ta & tb) / max(len(ta), 1)

def classify_failure(row) -> str:
    query = row['query']
    ground_truth = row['ground_truth']
    contexts = row['retrieved_contexts']
    if not contexts:
        return 'no_retrieval'
    
    combined = ' '.join(contexts)
    # vocabulary_mismatch: low query-context overlap despite enough context volume
    avg_q_overlap = np.mean([token_overlap(query, c) for c in contexts])
    gt_coverage = token_overlap(ground_truth, combined)
    
    if avg_q_overlap < 0.15:
        return 'vocabulary_mismatch'
    # information_split: ground truth tokens spread across multiple contexts, none covers >60%
    per_ctx_gt = [token_overlap(ground_truth, c) for c in contexts]
    if max(per_ctx_gt) < 0.4 and gt_coverage > 0.5:
        return 'information_split'
    # right_topic_wrong_detail: contexts overlap with query well but not with ground truth
    if avg_q_overlap > 0.3 and gt_coverage < 0.3:
        return 'right_topic_wrong_detail'
    # query_ambiguity: high overlap with contexts but low precision score
    if avg_q_overlap > 0.25 and row['context_precision'] < 0.35:
        return 'query_ambiguity'
    return 'wrong_chunk'

failing['failure_type'] = failing.apply(classify_failure, axis=1)
failing['failure_type'].value_counts()

## 3. Failure Distribution by Experiment

In [ ]:
failure_matrix = (
    failing.groupby(['experiment_id', 'failure_type'])
    .size()
    .unstack(fill_value=0)
)

ax = failure_matrix.plot(kind='bar', stacked=True, figsize=(12, 6))
ax.set_title('Retrieval Failure Types by Experiment')
ax.set_xlabel('Experiment')
ax.set_ylabel('Failing queries')
ax.legend(loc='upper right', bbox_to_anchor=(1.15, 1))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Key Insight: Which Strategies Fix Which Failure Types?

In [ ]:
# Compute failure rate per type per experiment
total_per_exp = query_df.groupby('experiment_id').size().rename('total')
failure_rates = (
    failing.groupby(['experiment_id', 'failure_type'])
    .size()
    .div(total_per_exp, level='experiment_id')
    .unstack(fill_value=0)
    .round(3)
)
print('Failure rates (fraction of queries):')
failure_rates

In [ ]:
failure_records = failing[['experiment_id', 'query_id', 'query', 'ground_truth',
                             'context_precision', 'context_recall', 'failure_type']].to_dict('records')
save_output(failure_records, 'retrieval_failures.json')

failure_summary = failure_rates.reset_index().to_dict('records')
save_output(failure_summary, 'failure_rates_by_experiment.json')
print('Saved.')